In [ ]:
import numpy as np
#%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd
#import seaborn as sns
import matplotlib.dates as mdates
import calendar
plt.rcParams['figure.dpi'] = 100
plt.rcParams['lines.markersize'] = 4
plt.rcParams['font.size'] = 14
plt.rcParams['axes.prop_cycle'] = plt.cycler('color', 'brgmyk')


In [ ]:
data_68 = pd.read_csv("monthly_n2o_68.csv")
data_46 = pd.read_csv("monthly_n2o_46.csv")
data_32 = pd.read_csv("monthly_n2o_32.csv")
data_22 = pd.read_csv("monthly_n2o_22.csv")

In [ ]:
o3_68 = pd.read_csv("monthly_o3_68.csv")
o3_46 = pd.read_csv("monthly_o3_46.csv")
o3_32 = pd.read_csv("monthly_o3_32.csv")
o3_22 = pd.read_csv("monthly_o3_22.csv")

In [ ]:
n2o_dataframes = [data_68, data_46, data_32, data_22]
o3_dataframes = [o3_68, o3_46, o3_32, o3_22]
altitudes = [68, 46, 32, 22]


In [ ]:
# Keep only months with at least 5 valid measurements, as in lat_zones_v6.ipynb and the Methods
# (added; remove this cell to reproduce the earlier figures)
n2o_dataframes = [df[df["count"] >= 5] for df in n2o_dataframes]
o3_dataframes = [df[df["count"] >= 5] for df in o3_dataframes]

In [ ]:
data_68

In [ ]:
# reversing altitudes for plotting
altitudes = altitudes[::-1]
n2o_dataframes = n2o_dataframes[::-1]
o3_dataframes = o3_dataframes[::-1]

In [ ]:
def add_detrended(df, date_column='date', group_col='location', 
                           value_col='mean_concentration', detrend_col='detrended_concentration'):
    """
    Adds a new column to the DataFrame that contains the values in value_col detrended
    by a linear fit performed separately for each group in group_col. The detrended values 
    are anchored at the center of the time series (so that the data remain on an absolute scale).
    
    NaN values in the value_col are ignored during the detrending calculation and remain NaN 
    in the resulting detrended column.
    
    Parameters:
      - df (DataFrame): Input DataFrame containing at least the date, group, and value columns.
      - date_column (str): Column name for the date. (Dates will be converted to datetime if needed.)
      - group_col (str): Column name on which to group the data (e.g. 'location').
      - value_col (str): Column name of the values to detrend (e.g. 'mean_concentration').
      - detrend_col (str): Name of the new column that will store the detrended values.
    
    Returns:
      - DataFrame: A copy of the original DataFrame with an additional column for detrended values.
    """
    # Create a copy of the dataframe to avoid modifying the original data.
    df = df.copy()
    
    # Ensure the date column is in datetime format.
    df[date_column] = pd.to_datetime(df[date_column])
    
    # Sort the DataFrame by group and date.
    df.sort_values([group_col, date_column], inplace=True)
    
    # Create the new column initialized with NaNs.
    df[detrend_col] = np.nan
    
    # Process each group separately.
    for loc, group in df.groupby(group_col):
        # Identify the valid (non-NaN) values.
        valid_mask = group[value_col].notna()
        if valid_mask.sum() < 2:
            # If there are fewer than 2 valid points, we can't fit a trend; skip this group.
            continue

        # Work only with the valid entries.
        sub = group.loc[valid_mask]
        # Convert dates to ordinal numbers for fitting.
        time_ord = sub[date_column].apply(lambda x: x.toordinal()).values
        values = sub[value_col].values
        
        # Fit a linear trend: values = a * time + b.
        coeffs = np.polyfit(time_ord, values, 1)
        
        # Define a baseline using the mean ordinal value so that the detrended data are 
        # adjusted relative to the center of the time period.
        t_center = time_ord.mean()
        baseline = np.polyval(coeffs, t_center)
        
        # Calculate the detrended values.
        # This subtracts the deviation of the trend from its baseline.
        detrended_values = values - (np.polyval(coeffs, time_ord) - baseline)
        
        # Assign the detrended values back to the original DataFrame.
        df.loc[sub.index, detrend_col] = detrended_values
    
    return df


In [ ]:
# Define a mapping from internal location names to reader-friendly names.
display_names = {
    'bashkortostan': 'Bashkortostan',
    'bozeman': 'Montana',
    'brunei': 'Borneo',
    'california': 'California',
    'catalonia': 'Catalonia',
    'colombia': 'Colombia',
    'estonia': 'Estonia',
    'finland': 'Finland',
    'florianopolis': 'Santa Catarina',
    'florida': 'Florida',
    'france': 'France',
    'french_guiana': 'French Guiana',
    'huntingdon': 'Southern Québec',
    'iceland_e': 'Iceland (E)',
    'iceland_w': 'Iceland (W)',
    'khabarovsk': 'Khabarovsk',
    'kongo': 'Congo',
    'kyrgyzstan': 'Kyrgyzstan',
    'mexico': 'Xochimilco, Mexico City',
    'morocco': 'Morocco',
    'mukhrino': 'Mukhrino',
    'myanmar': 'Myanmar',
    'nz_n': 'New Zealand (N)',
    'nz_s': 'New Zealand (S)',
    'pantanal': 'Pantanal',
    'quistococha': 'Peruvian Amazon',
    'romania': 'Romania',
    'taiwan': 'Taiwan',
    'tarapoto': 'Tarapoto',
    'tasmania': 'Tasmania',
    'tierra_del_fuego': 'Tierra del Fuego',
    'uganda_e': 'Uganda (E)',
    'uganda_n': 'Uganda (N)',
    'uganda_s': 'Uganda (S)',
    'wales': 'Wales'
    # Add more mappings as needed.
}

In [ ]:
# Initialize a new list to hold the updated DataFrames.
n2o_detrended = []

# Loop over each DataFrame, apply the detrending function, and append the result to the new list.
for df in n2o_dataframes:
    new_df = add_detrended(df, 
                           date_column='date', 
                           group_col='location', 
                           value_col='mean_concentration', 
                           detrend_col='detrended_concentration')
    n2o_detrended.append(new_df)

In [ ]:
# Initialize a new list to hold the updated DataFrames.
o3_detrended = []

# Loop over each DataFrame, apply the detrending function, and append the result to the new list.
for df in o3_dataframes:
    new_df = add_detrended(df, 
                           date_column='date', 
                           group_col='location', 
                           value_col='mean_concentration', 
                           detrend_col='detrended_concentration')
    o3_detrended.append(new_df)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import calendar
from scipy import stats as ss


def location_monthly_stats(df, value_col='detrended_concentration', date_col='date',
                           location_col='location', confidence=0.95, anomaly=None):
    """Mean and CI per location and calendar month. anomaly='abs' subtracts each location's
    annual mean (mean of its 12 monthly means); 'pct' expresses that difference in % of it."""
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])
    df['month'] = df[date_col].dt.month
    st = (df.groupby([location_col, 'month'])[value_col]
            .agg(['mean', 'std', 'count']).reset_index())
    alpha = 1 - confidence
    st['ci_half'] = st.apply(
        lambda r: ss.t.ppf(1 - alpha/2, df=r['count']-1) * r['std'] / (r['count']**0.5)
        if r['count'] > 1 else 0, axis=1)
    if anomaly in ('abs', 'pct'):
        annual = st.groupby(location_col)['mean'].transform('mean')
        st['mean'] = st['mean'] - annual
        if anomaly == 'pct':
            st['mean'] = 100 * st['mean'] / annual
            st['ci_half'] = 100 * st['ci_half'] / annual
    elif anomaly is not None:
        raise ValueError("anomaly must be None, 'abs' or 'pct'")
    st['ci_lower'] = st['mean'] - st['ci_half']
    st['ci_upper'] = st['mean'] + st['ci_half']
    return st


def shared_anomaly_limits(n2o_dfs, o3_dfs, locations=None, anomaly='abs', pad=1.05, **kw):
    """One symmetric y-range per gas covering all levels and the given locations (all if None),
    so the NH, tropics and SH figures can use the same scale."""
    lims = {}
    for sp, dfs in (('N2O', n2o_dfs), ('O3', o3_dfs)):
        m = 0.0
        for df in dfs:
            st = location_monthly_stats(df, anomaly=anomaly, **kw)
            if locations is not None:
                st = st[st[kw.get('location_col', 'location')].isin(locations)]
            m = max(m, st['ci_lower'].abs().max(), st['ci_upper'].abs().max())
        lims[sp] = (-round(float(m) * pad, 3), round(float(m) * pad, 3))
    return lims


def plot_monthly_means_by_location_across_period(
    n2o_dfs: list,
    o3_dfs: list,
    location_inds: list,
    display_names: dict,
    altitudes: list = ('22 hPa', '32 hPa', '46 hPa', '68 hPa'),
    value_col: str = 'detrended_concentration',
    date_col: str = 'date',
    location_col: str = 'location',
    confidence: float = 0.95,
    palette: list = None,
    loc_ci_alpha: dict = None,
    show_errorbar: bool = False,
    save_path=None,
    save_dpi=300,
    anomaly: str = None,
    ylims: dict = None,
):
    """
    Plots mean detrended concentrations and 95% CI by month-of-year for chosen locations
    across the altitudes (rows) and two species (cols: N2O left, O3 right).

    anomaly=None  : absolute values, each subplot scaled to its own data (as before)
    anomaly='abs' : each location minus its own annual mean (ppbv / ppmv)
    anomaly='pct' : the same as a percentage of the annual mean
    ylims={'N2O': (lo, hi), 'O3': (lo, hi)} fixes the y-limits per species; in anomaly mode they
    default to one symmetric range per species for the whole figure. Pass the output of
    shared_anomaly_limits() to give several figures the same scale.

    Colours repeat after the palette runs out, so each further pass through it uses a different
    line style (solid, dashed, dotted), keeping every location distinguishable in the legend.
    Returns the y-limits used.
    """
    default_palette = ['#377eb8', '#ff7f00', '#4daf4a', '#f781bf',
                       '#a65628', '#984ea3', '#999999', '#e41a1c', '#dede00']
    palette = palette or default_palette
    linestyles = ['-', '--', ':']
    loc_ci_alpha = loc_ci_alpha or {}
    default_alpha = 0.3
    species = ('N2O', 'O3')
    kw = dict(value_col=value_col, date_col=date_col, location_col=location_col, confidence=confidence)

    stats_n2o = [location_monthly_stats(df, anomaly=anomaly, **kw) for df in n2o_dfs]
    stats_o3 = [location_monthly_stats(df, anomaly=anomaly, **kw) for df in o3_dfs]

    ylims = dict(ylims or {})
    if anomaly in ('abs', 'pct'):
        for sp, lst in zip(species, (stats_n2o, stats_o3)):
            if sp not in ylims:
                sel = [st[st[location_col].isin(location_inds)] for st in lst]
                m = max(max(s['ci_lower'].abs().max(), s['ci_upper'].abs().max()) for s in sel)
                ylims[sp] = (-round(float(1.05 * m), 3), round(float(1.05 * m), 3))

    month_labels = list(calendar.month_abbr)[1:]
    n_alt = len(altitudes)
    fig, axes = plt.subplots(nrows=n_alt, ncols=2, figsize=(14, 4 * n_alt), sharex=True, squeeze=False)

    for i, alt in enumerate(altitudes):
        for j, stats_list in enumerate((stats_n2o, stats_o3)):
            ax = axes[i, j]
            for idx, loc in enumerate(location_inds):
                display = display_names.get(loc, loc)
                df_stats = stats_list[i]
                grp = df_stats[df_stats[location_col] == loc].sort_values('month')
                if grp.empty:
                    continue
                color = palette[idx % len(palette)]
                ls = linestyles[(idx // len(palette)) % len(linestyles)]
                a_ci = loc_ci_alpha.get(loc, default_alpha)

                ax.fill_between(grp['month'], grp['ci_lower'], grp['ci_upper'],
                                color=color, alpha=a_ci, zorder=1)
                if show_errorbar:
                    ax.errorbar(grp['month'], grp['mean'], yerr=grp['ci_half'],
                                fmt='none', ecolor=color, alpha=a_ci, capsize=3, zorder=2)
                ax.plot(grp['month'], grp['mean'], color=color, linestyle=ls, label=display, zorder=3)

            ax.set_title(f"{alt}")
            ax.grid(True)
            if species[j] in ylims:
                ax.set_ylim(*ylims[species[j]])
            if anomaly in ('abs', 'pct'):
                ax.axhline(0, color='0.4', lw=0.8, zorder=0)
            unit = {'abs': ('ppbv', 'ppmv'), 'pct': ('%', '%')}.get(anomaly, ('ppbv', 'ppmv'))[j]
            gas = 'N₂O' if j == 0 else 'O₃'
            ax.set_ylabel(f"{gas} anomaly ({unit})" if anomaly else f"{gas} ({unit})")
            if i < n_alt - 1:
                ax.tick_params(labelbottom=False)

    for j in range(2):
        axes[-1, j].set_xticks(range(1, 13))
        axes[-1, j].set_xticklabels(month_labels, rotation=45)

    handles, labels = axes[0, 0].get_legend_handles_labels()
    legend_rows = -(-len(labels) // 5)
    fig.tight_layout(rect=[0, 0.015 + 0.014 * legend_rows * 4 / n_alt, 1, 1])
    fig.legend(handles, labels, loc='lower center',
               ncol=5, fontsize=10, bbox_to_anchor=(0.5, 0.005))
    if save_path:
        plt.savefig(save_path, dpi=save_dpi)
    plt.show()
    return ylims

In [ ]:
locations_sh = ['tierra_del_fuego', 'tasmania', 'nz_n', 'nz_s']
locations_eq = ['brunei', 'colombia', 'florianopolis', 'florida', 'french_guiana', 'kongo', 'mexico', 'myanmar', 'pantanal', 
                'quistococha', 'taiwan', 'tarapoto', 'uganda_e', 'uganda_n', 'uganda_s']
locations_nh = ['bashkortostan', 'bozeman', 'california', 'catalonia', 'estonia', 'finland',
                'france', 'huntingdon', 'iceland_e', 'iceland_w', 'khabarovsk', 
                'kyrgyzstan', 'morocco', 'mukhrino', 'romania', 'wales']
# Florida (26.4°N) moved from locations_nh to locations_eq: it is in the tropics zone in zone_dict and Table S6

## Seasonal anomalies per location
Each location minus its own annual mean. By default each figure gets one y-range per gas, shared by its four levels and all its locations. To put the NH, tropics and SH figures on the same scale instead, switch `lims` in the next cell (the tropical curves then get much flatter).

In [ ]:
lims = None   # one scale per figure
# lims = shared_anomaly_limits(n2o_detrended, o3_detrended,
#                              locations=locations_nh + locations_eq + locations_sh, anomaly='abs')
# print(lims)

In [ ]:
plot_monthly_means_by_location_across_period(
    n2o_detrended, o3_detrended,
    location_inds=locations_nh, display_names=display_names,
    show_errorbar=True, anomaly='abs', ylims=lims,
    save_path='nh_all_seasonal_anomaly_v6.png'
)

In [ ]:
plot_monthly_means_by_location_across_period(
    n2o_detrended, o3_detrended,
    location_inds=locations_eq, display_names=display_names,
    show_errorbar=True, anomaly='abs', ylims=lims,
    save_path='eq_all_seasonal_anomaly_v6.png'
)

In [ ]:
plot_monthly_means_by_location_across_period(
    n2o_detrended, o3_detrended,
    location_inds=locations_sh, display_names=display_names,
    show_errorbar=True, anomaly='abs', ylims=lims,
    save_path='sh_all_seasonal_anomaly_v6.png'
)

## Absolute values (earlier version of the figures)

In [ ]:
plot_monthly_means_by_location_across_period(
    n2o_detrended,
    o3_detrended,
    location_inds=locations_sh,
    display_names=display_names,
    loc_ci_alpha={'SiteC':0.15},
    show_errorbar=True,
    save_path = 'sh_all_seasonal_v6.png'
)


In [ ]:
plot_monthly_means_by_location_across_period(
    n2o_detrended,
    o3_detrended,
    location_inds=locations_eq,
    display_names=display_names,
    loc_ci_alpha={'SiteC':0.15},
    show_errorbar=True,
    save_path = 'eq_all_seasonal_v6.png'
)


In [ ]:
plot_monthly_means_by_location_across_period(
    n2o_detrended,
    o3_detrended,
    location_inds=locations_nh,
    display_names=display_names,
    loc_ci_alpha={'SiteC':0.15},
    show_errorbar=True,
    save_path = 'nh_all_seasonal.png'
)
